In [1]:
import pandas as pd
import re
df = pd.read_csv("Source 2.csv")

df = df.drop(columns=["published_at"])

print(df.columns)
print(df.head())

# 4) Save to a new CSV without 'published_at'
df.to_csv("top_600_youtube_videos_2023.csv", index=False)

Index(['title', 'duration', 'view_count', 'like_count', 'comment_count'], dtype='object')
                                               title duration  view_count  \
0       Let's see if you dare next time #Shorts #017    PT18S   246200629   
1                           [6번] 식당 업소용 음식포장 비닐랩 절단기    PT15S   160984366   
2   Khaby Lame react epic moments #shorts #kabhylame    PT19S    96455747   
3      Polina Knoroz's Cute Moment in Sports #shorts    PT35S    91902366   
4  Salute to indian army🇮🇳❤️💯 #army #trending #vi...    PT30S    85157682   

   like_count  comment_count  
0   1572774.0          942.0  
1   1472323.0          678.0  
2   2312319.0         2203.0  
3   1397067.0          930.0  
4   1973374.0            0.0  


In [2]:
pattern = re.compile(
    r"PT"              # starts with PT
    r"(?:(\d+)H)?"     # hours
    r"(?:(\d+)M)?"     # minutes
    r"(?:(\d+)S)?"     # seconds
)

def iso_to_seconds(s):
    if not isinstance(s, str):
        return None
    m = pattern.fullmatch(s)
    if not m:
        return None
    h = int(m.group(1)) if m.group(1) else 0
    m_ = int(m.group(2)) if m.group(2) else 0
    s_ = int(m.group(3)) if m.group(3) else 0
    return h*3600 + m_*60 + s_

df["duration_seconds"] = df["duration"].apply(iso_to_seconds)

df.to_csv("top_600_youtube_videos_2023.csv", index=False)

In [3]:
df

,title,duration,view_count,like_count,comment_count,duration_seconds
0,Let's see if you dare next time #Shorts #017,PT18S,246200629,1572774.0,942.0,18.0
1,[6번] 식당 업소용 음식포장 비닐랩 절단기,PT15S,160984366,1472323.0,678.0,15.0
2,Khaby Lame react epic moments #shorts #kabhylame,PT19S,96455747,2312319.0,2203.0,19.0
3,Polina Knoroz's Cute Moment in Sports #shorts,PT35S,91902366,1397067.0,930.0,35.0
4,Salute to indian army🇮🇳❤️💯 #army #trending #vi...,PT30S,85157682,1973374.0,0.0,30.0
...,...,...,...,...,...,...
596,White People Hate Black Hair?,PT1M,99052,11291.0,1210.0,60.0
597,Biden White House is 'stonewalling' this inves...,PT5M27S,94648,2690.0,964.0,327.0
598,[다시보기] 백현동 발언 허위라도…ㅣ2023년 12월 11일 김진의 더라방,PT42M49S,93376,7401.0,169.0,2569.0
599,[2023년 12월 11일 월요일 오전 8시 생방송] 김기현 거취 결단 요구 봇물,PT2H56M59S,89526,15254.0,152.0,10619.0


In [4]:
df = df.drop(columns=["duration"])

print(df.columns)
df.head()

Index(['title', 'view_count', 'like_count', 'comment_count',
       'duration_seconds'],
      dtype='object')


,title,view_count,like_count,comment_count,duration_seconds
0,Let's see if you dare next time #Shorts #017,246200629,1572774.0,942.0,18.0
1,[6번] 식당 업소용 음식포장 비닐랩 절단기,160984366,1472323.0,678.0,15.0
2,Khaby Lame react epic moments #shorts #kabhylame,96455747,2312319.0,2203.0,19.0
3,Polina Knoroz's Cute Moment in Sports #shorts,91902366,1397067.0,930.0,35.0
4,Salute to indian army🇮🇳❤️💯 #army #trending #vi...,85157682,1973374.0,0.0,30.0


In [5]:
print(f"Original rows: {len(df)}")

df = df.dropna()
print(f"After drop null values: {len(df)}")


df = df.drop_duplicates()
print(f"After drop_duplicates {len(df)}")

# Optional: save the fully cleaned data
df.to_csv("top_600_youtube_videos_2023_clean_no_null_no_dup.csv", index=False)

Original rows: 601
After drop null values: 584
After drop_duplicates 463


In [6]:
df["like_count"] = df["like_count"].fillna(0).astype(int)
df["comment_count"] = df["comment_count"].fillna(0).astype(int)
df["duration_seconds"] = df["duration_seconds"].fillna(0).astype(int)

print(df.dtypes.head())

title               object
view_count           int64
like_count           int32
comment_count        int32
duration_seconds     int32
dtype: object


In [7]:
df

,title,view_count,like_count,comment_count,duration_seconds
0,Let's see if you dare next time #Shorts #017,246200629,1572774,942,18
1,[6번] 식당 업소용 음식포장 비닐랩 절단기,160984366,1472323,678,15
2,Khaby Lame react epic moments #shorts #kabhylame,96455747,2312319,2203,19
3,Polina Knoroz's Cute Moment in Sports #shorts,91902366,1397067,930,35
4,Salute to indian army🇮🇳❤️💯 #army #trending #vi...,85157682,1973374,0,30
...,...,...,...,...,...
596,White People Hate Black Hair?,99052,11291,1210,60
597,Biden White House is 'stonewalling' this inves...,94648,2690,964,327
598,[다시보기] 백현동 발언 허위라도…ㅣ2023년 12월 11일 김진의 더라방,93376,7401,169,2569
599,[2023년 12월 11일 월요일 오전 8시 생방송] 김기현 거취 결단 요구 봇물,89526,15254,152,10619


In [8]:
print(f"Original rows: {len(df)}")

df = df[(df['like_count'] <= df['view_count']) & 
        (df['comment_count'] <= df['view_count'])]
 #if number of likes or comments is greater than views ->impossible values
print(f"After removing impossible values: {len(df)}")

min_ratio = 0.0001 #as the youtube average ratio is 0.01 so 0.0001 (which is applied) is very suspicious 
df = df[
    ((df['like_count'] / df['view_count']) >= min_ratio) |
    ((df['comment_count'] / df['view_count']) >= min_ratio)
]
 #if number of likes or comments doesn't reach the minimum ratio then it is removed
print(f"After removing suspicious low engagement: {len(df)}")

df.to_csv("top_600_youtube_videos_2023_clean_no_null_no_dup_wih_ratio.csv", index=False)

Original rows: 463
After removing impossible values: 463
After removing suspicious low engagement: 451
